# Comparativo entre Modelos de Predição — Base Glicemia

Este notebook treina 7 classificadores para prever o nível de **GLICEMIA** (0 = Abaixo, 1 = Normal, 2 = Acima) a partir de INSULINA, KCAL, CARB, SONO e padel.

> Os outputs abaixo (tabelas, matrizes de confusão, ranking) são os resultados **reais** obtidos na sua execução — deixados aqui "crus", exatamente como saíram, seguidos da explicação do que cada um significa.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

# Modelos de classificação
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

In [2]:
# 1. Carregar os dados
url = 'https://raw.githubusercontent.com/alexandrezamberlan/tias/refs/heads/main/3_predicao_previsao_codigos_exemplos/glicose_data.csv'
df = pd.read_csv(url)
df

         DIA   ANO AC|DC GLICEMIA  INSULINA         KCAL         CARB  SONO  \
0     Quinta  2012    ac   Normal         6       Abaixo        Acima   4.0   
1      Sexta  2012    ac   Normal         6  Recomendado        Acima   4.0   
2     Sabado  2012    ac   Normal         6  Recomendado        Acima   4.0   
3    Domingo  2012    ac    Acima         6       Abaixo        Acima   5.0   
4    Segunda  2012    ac   Normal         6  Recomendado        Acima   5.0   
..       ...   ...   ...      ...       ...          ...          ...   ...   
719  Domingo  2014    ac    Acima        12  Recomendado        Acima   5.0   
720  Segunda  2014    ac   Normal        12  Recomendado       Abaixo   5.0   
721    Terca  2014    ac   Normal        12  Recomendado        Acima   5.0   
722   Quarta  2014    ac   Normal        12  Recomendado  Recomendado   3.0   
723   Quinta  2014    ac   Normal        12  Recomendado        Acima   4.0   

     padel  musculacao_R  ...  corrida  caminhada  

**Leitura do resultado:** 724 linhas e 22 colunas, cobrindo 2012 a 2014. Além das colunas usadas no modelo, existem outras atividades físicas (corrida, caminhada, tênis, sauna, bike, natação, elíptico, vôlei de areia) e duas colunas sem nome (`Unnamed: 20`, `Unnamed: 21`, só `NaN`) que ficam de fora do modelo.

In [3]:
#no df preciso trocar na coluna KCAL os valores Abaixo para 0; Recomendado para 1; Acima para 2
df['KCAL'] = df['KCAL'].replace({'Abaixo': 0, 'Recomendado': 1, 'Acima': 2})

#no df preciso trocar na coluna CARB os valores Abaixo para 0; Recomendado para 1; Acima para 2
df['CARB'] = df['CARB'].replace({'Abaixo': 0, 'Recomendado': 1, 'Acima': 2})

#no df preciso trocar na coluna GLICEMIA os valores Abaixo para 0; Normal para 1; Acima para 2
df['GLICEMIA'] = df['GLICEMIA'].replace({'Abaixo': 0, 'Normal': 1, 'Acima': 2})

In [4]:
# 2. Features e variável alvo
features = ['INSULINA', 'KCAL', 'CARB', 'SONO', 'padel']
target = 'GLICEMIA'

X = df[features]
y = df[target]
y

0      1
1      1
2      1
3      2
4      1
      ..
719    2
720    1
721    1
722    1
723    1
Name: GLICEMIA, Length: 724, dtype: int64

**Leitura do resultado:** confirma que a coluna `GLICEMIA` já está codificada em 0/1/2 (0 = Abaixo, 1 = Normal, 2 = Acima), com 724 valores no total (ainda sem remover os `NaN`).

In [5]:
#no X preciso eliminar as linhas com colunas NaN
X = X.dropna()

#no y preciso eliminar as linhas com colunas NaN
y = y.dropna()

#no y preciso eliminar a linha 446
y = y.drop(446)

In [6]:
# 3. Divisão treino/teste com estratificação
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

In [7]:
# 4. Padronizar (necessário para SVM e KNN)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [8]:
# 5. Modelos de classificação
modelos = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(),
    'Random Forest': RandomForestClassifier(),
    'KNN': KNeighborsClassifier(),
    'Naive Bayes': GaussianNB(),
    'SVM': SVC(),
    'Gradient Boosting': GradientBoostingClassifier()
}

In [9]:
# 6. Avaliar cada modelo
resultados = []

print("Avaliação dos Modelos:\n")

for nome, modelo in modelos.items():
    modelo.fit(X_train, y_train)
    y_pred = modelo.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)

    resultados.append((nome, acc, f1))

    #ordenar
    print(f"Modelo: {nome}")
    print(f"Acurácia: {acc:.4f}")
    print(f"F1-Score (Macro): {f1:.4f}")
    print("Relatório de Classificação:")
    print(classification_report(y_test, y_pred, zero_division=0))
    print("Matriz de Confusão:")
    print(confusion_matrix(y_test, y_pred))
    print("-" * 60)

Avaliação dos Modelos:

Modelo: Logistic Regression
Acurácia: 0.7558
F1-Score (Macro): 0.2870
Relatório de Classificação:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         7
           1       0.76      0.99      0.86       166
           2       0.00      0.00      0.00        44

    accuracy                           0.76       217
   macro avg       0.25      0.33      0.29       217
weighted avg       0.58      0.76      0.66       217

Matriz de Confusão:
[[  0   7   0]
 [  0 164   2]
 [  0  44   0]]
------------------------------------------------------------
Modelo: Decision Tree
Acurácia: 0.7005
F1-Score (Macro): 0.2973
Relatório de Classificação:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         7
           1       0.76      0.90      0.82       166
           2       0.13      0.05      0.07        44

    accuracy                           0.70       217
   mac

**Como ler cada bloco acima ("cru"):**
- **Acurácia**: % total de acertos, sem diferenciar classe.
- **F1-Score (Macro)**: média do F1 das 3 classes, todas com peso igual — não deixa a classe majoritária "esconder" o desempenho ruim nas minoritárias.
- **Relatório de Classificação**: `support` é o número de amostras reais de cada classe no teste (7 / 166 / 44); `precision` = das vezes que o modelo previu essa classe, quantas estavam certas; `recall` = das amostras reais dessa classe, quantas o modelo conseguiu encontrar.
- **Matriz de Confusão**: linhas = classe real, colunas = classe prevista. A diagonal principal (topo-esquerda a baixo-direita) são os acertos; tudo fora da diagonal é erro. Ex.: na matriz da Logistic Regression, a linha 0 é `[0, 7, 0]` — as 7 amostras reais da classe 0 foram todas previstas como classe 1 (nenhum acerto).

## Como ler a matriz de confusão, linha por linha

Ela é uma tabela 3x3 (3 classes: 0 = Abaixo, 1 = Normal, 2 = Acima):

- **Cada linha** = a classe **real** da amostra (o que de fato aconteceu)
- **Cada coluna** = a classe que o **modelo previu**
- **Célula [linha, coluna]** = quantas amostras daquela classe real caíram naquela previsão

```
                    Previsto 0   Previsto 1   Previsto 2
Real 0 (Abaixo)     [   a            b            c    ]
Real 1 (Normal)     [   d            e            f    ]
Real 2 (Acima)      [   g            h            i    ]
```

A **diagonal principal** (a, e, i) são os **acertos**. Tudo fora da diagonal é **erro**.

### Exemplo: Logistic Regression

```
[[  0   7   0]
 [  0 164   2]
 [  0  44   0]]
```

**Linha 1 → `[0, 7, 0]`** = classe real **0 (Abaixo)**, total 7 amostras (0+7+0=7). O modelo:
- previu "0" → **0 vezes** (não acertou nenhuma)
- previu "1" (Normal) → **7 vezes** (errou todas, jogando tudo pra classe majoritária)
- previu "2" → 0 vezes

**Linha 2 → `[0, 164, 2]`** = classe real **1 (Normal)**, total 166 amostras (0+164+2=166). O modelo:
- previu "0" → 0 vezes
- previu "1" (certo) → **164 vezes** (acertou quase tudo)
- previu "2" → 2 vezes (só errou 2)

**Linha 3 → `[0, 44, 0]`** = classe real **2 (Acima)**, total 44 amostras. O modelo:
- previu "0" → 0 vezes
- previu "1" → **44 vezes** (errou 100% — jogou tudo pra "Normal" de novo)
- previu "2" (certo) → 0 vezes

Esse modelo, na prática, só sabe dizer "Normal". Acerta bem a classe 1 porque ela é 77% do teste, mas é cego pras classes 0 e 2.

### Comparando com o SVM (melhor do ranking)

```
[[  0   7   0]
 [  0 165   1]
 [  0  42   2]]
```

- **Linha 1 (real=0)**: `[0, 7, 0]` → mesma coisa, 0 acertos na classe 0.
- **Linha 2 (real=1)**: `[0, 165, 1]` → acerta 165 de 166, quase perfeito.
- **Linha 3 (real=2)**: `[0, 42, 2]` → das 44 amostras reais "Acima", acertou só **2**, errou 42 chamando de "Normal".

Olhando agora pela **coluna** 2 (previsões "Acima") em vez da linha: o modelo só previu "2" poucas vezes no total, e a maioria das vezes que arriscou, acertou — por isso a *precision* da classe 2 deu 0,67 no relatório. Só que ele arrisca muito pouco, por isso o *recall* é baixíssimo (0,05): detecta quase nada dos casos reais.

### Resumo da lógica

| Você quer saber... | Olhe... |
|---|---|
| Quantas amostras reais de uma classe existiam | Some a **linha** inteira |
| Onde o modelo está confundindo essa classe real | Veja pra quais colunas os erros da linha foram |
| Se o modelo é "confiável" quando prevê uma classe | Olhe a **coluna** — quantos dos previstos ali eram realmente daquela classe |
| Quantas vezes o modelo nunca acerta uma classe | Veja se a diagonal daquela linha/coluna é 0 |

No seu caso, o padrão se repete em quase todos os 7 modelos: a **linha 0 (real=Abaixo)** nunca tem acerto na diagonal — o modelo simplesmente não tem exemplos suficientes (só 7 no teste) pra aprender a reconhecer essa classe.

In [10]:
# 7. Ranking final por F1-Score Macro
resultados.sort(key=lambda x: x[2], reverse=True)

print("Ranking Final dos Modelos:")
print(f"{'Modelo':<25} {'Acurácia':<10} {'F1-Score (Macro)':<15}")
print("-" * 50)
for nome, acc, f1 in resultados:
    print(f"{nome:<25} {acc:<10.4f} {f1:<15.4f}")

Ranking Final dos Modelos:
Modelo                    Acurácia   F1-Score (Macro)
--------------------------------------------------
SVM                       0.7696     0.3178         
KNN                       0.7097     0.3116         
Gradient Boosting         0.7512     0.3116         
Random Forest             0.7051     0.2977         
Decision Tree             0.7005     0.2973         
Logistic Regression       0.7558     0.2870         
Naive Bayes               0.7327     0.2819         


## Análise dos resultados

### 1. O dataset de teste está desbalanceado
O `support` do relatório de classificação mostra a divisão real das 217 amostras de teste:

| Classe | Significado | support | % do teste |
|---|---|---|---|
| 0 | Abaixo | 7 | ~3% |
| 1 | Normal | 166 | ~77% |
| 2 | Acima | 44 | ~20% |

Quase 8 em cada 10 dias o paciente está com glicemia **Normal**. Esse é o ponto-chave para interpretar todos os números acima.

### 2. Por que a acurácia sozinha engana
Um modelo que **sempre chuta "Normal"** já acertaria ~77% sem aprender nada. É o que Logistic Regression (75,6% acurácia) e Naive Bayes (73,3%) fazem na prática — `precision` e `recall` = 0,00 nas classes 0 e 2, e a matriz de confusão confirma: toda a coluna do meio (classe 1 prevista) recebe as amostras das outras classes.

### 3. Por que o F1 macro é o critério certo
Ele penaliza modelos que ignoram classes minoritárias — por isso Logistic Regression e Naive Bayes, mesmo com acurácia mais alta, caem para as últimas posições do ranking.

### 4. Leitura do ranking final
- **1º SVM (F1 0,3178)**: melhor equilíbrio geral. Na classe 2 tem `precision` 0,67 — quando arrisca prever "Acima", costuma acertar — mas `recall` 0,05, ou seja, detecta muito pouco dos casos reais dessa classe.
- **2º KNN e Gradient Boosting (empate, F1 0,3116)**: únicos, junto com Decision Tree e Random Forest, que conseguem prever as 3 classes ao mesmo tempo (nem que seja pouco).
- **6º e 7º Logistic Regression e Naive Bayes**: acurácia "de fachada" — nunca preveem as classes 0 e 2.

**Nenhum dos 7 modelos acerta a classe 0 ("Abaixo")** — em todas as matrizes de confusão a primeira coluna (previsões da classe 0) está zerada. Com só 7 amostras no teste (e poucas dezenas no treino), não há dados suficientes para aprender esse padrão.

### 5. Conclusão prática
O gargalo não é o algoritmo escolhido — é o **desbalanceamento das classes** (sobretudo a classe 0) e possivelmente a fraca relação entre as features usadas e o nível de glicemia. Caminhos para melhorar: balanceamento (SMOTE, `class_weight='balanced'`), usar mais variáveis do dataset original (outras atividades físicas), ou agrupar 0 e 2 em uma única classe "fora do normal" se o objetivo prático for só detectar desvio da glicemia.